
  <img src="https://github.com/gefero/factor_data_tuto_NLP_SICSS/blob/main/imgs/logo_final_conjunto.png?raw=true" width="80%">


# Summer Institute in Computational Social Sciences - Buenos Aires 2026
# Taller: Procesamiento de Lenguaje Natural y polarización
# Clasificación de tweets con distintas representaciones: TF, TF-IDF y Word Embeddings

# Introducción

El objetivo de este notebook es comparar distintas formas de representar texto como *features* para un modelo de clasificación, usando como caso el dataset **HatEval** (SemEval-2019 Task 5), que ya vienen utilizando en las prácticas de este taller. El dataset viene dividido en tres splits, cada uno en su propio archivo dentro de `data/`:

- `hateval_train_df.csv`: split de **entrenamiento**. Lo usamos para ajustar (`fit`) cada modelo.
- `hateval_dev_df.csv`: split de **validación**. Lo usamos para elegir el mejor valor del hiperparámetro de regularización de cada modelo.
- `hateval_test_df.csv`: split de **test**. Lo usamos únicamente al final, para evaluar el desempeño de cada modelo sobre datos que no participaron ni del entrenamiento ni de la elección de hiperparámetros.

Cada fila de estos archivos es un tweet con las siguientes columnas relevantes:

- `id`: identificador del tweet
- `text`: el texto del tweet
- `language`: idioma del tweet (`en` o `es`)
- `HS`: 1 si el tweet contiene discurso de odio (*hate speech*), 0 en caso contrario
- `TR`, `AG`: otras anotaciones (agresividad, si el odio está dirigido a un individuo o a un grupo) que no vamos a usar acá

La tarea de clasificación va a ser predecir `HS` (discurso de odio sí/no) a partir del texto del tweet.

Como en la última sección vamos a usar **embeddings preentrenados en español** (SBWCE), y los tres splits tienen tweets tanto en inglés como en español, nos vamos a quedar solamente con los tweets en español (`language == 'es'`) de cada split para poder comparar las tres representaciones sobre los mismos datos.

Al igual que en el ejemplo con reseñas de Amazon (`cap0/ejemplo_clasificacion.ipynb`), vamos a entrenar en cada caso una regresión logística regularizada por LASSO (penalización L1), variando el hiperparámetro de regularización $C$ (a menor $C$, mayor regularización). A diferencia de ese ejemplo, acá **no** usamos validación cruzada (K-Fold) para elegir $C$: como ya contamos con un split de validación (`dev`) independiente, entrenamos cada candidato sobre `train` y elegimos el que mejor performa sobre `dev`. `test` queda completamente afuera de ese proceso, y solo se usa al final para reportar el desempeño de cada modelo ya elegido.

Vamos a comparar tres formas de vectorizar los tweets:

1. **TF** (*Term Frequency*, bolsa de palabras con conteos)
2. **TF-IDF** (*Term Frequency - Inverse Document Frequency*)
3. **Word embeddings preentrenados** (promedio de vectores de palabras)

In [ ]:
## Ejecutar para descargar los embeddings preentrenados en español (SBWCE)
!wget -P ./models https://cs.famaf.unc.edu.ar/~ccardellino/SBWCE/SBW-vectors-300-min5.bin.gz && gunzip ./models/SBW-vectors-300-min5.bin.gz
!pip install gensim
!git clone https://github.com/gefero/factor_data_tuto_NLP_SICSS.git

# TF con LASSO

## Preparación de los datos

Cargamos los tres splits de HatEval (`hateval_train_df.csv`, `hateval_dev_df.csv` y `hateval_test_df.csv`), nos quedamos en cada uno con los tweets en español y preprocesamos el texto:

1. Convertimos a minúsculas
2. Eliminamos URLs y menciones (`@usuario`), que son ruido propio de los tweets
3. Eliminamos signos de puntuación
4. Reemplazamos números por la palabra `DIGITO`
5. Eliminamos acentos y caracteres no ASCII

Para cada split, `X` va a ser el texto preprocesado y `y` la variable binaria `HS` (discurso de odio).

In [ ]:
# Importamos las librerías necesarias
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import re
import unicodedata
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train_path = './factor_data_tuto_NLP_SICSS/data/hateval_train_df.csv'
dev_path = './factor_data_tuto_NLP_SICSS/data/hateval_dev_df.csv'
test_path = './factor_data_tuto_NLP_SICSS/data/hateval_test_df.csv'

In [ ]:
# Función para preprocesar el texto
def preprocess_text(text):
    # Convertir a minúsculas
    text = text.lower()

    # Eliminar URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Eliminar menciones (@usuario)
    text = re.sub(r'@\w+', ' ', text)

    # Reemplazar puntuación
    text = re.sub(r'[^\w\s]', ' ', text)

    # Reemplazar números por 'DIGITO'
    text = re.sub(r'\d+', 'DIGITO', text)

    # Reemplazar caracteres no ASCII (acentos, etc.)
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')

    # Colapsar espacios múltiples
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Cargamos un split y nos quedamos con los tweets en español
def load_hateval_split(path):
    df = pd.read_csv(path)
    df = df[df['language'] == 'es'].reset_index(drop=True)
    df['text_clean'] = df['text'].apply(preprocess_text)
    return df

train_df = load_hateval_split(train_path)
dev_df = load_hateval_split(dev_path)
test_df = load_hateval_split(test_path)

# Preparamos las variables para el modelo
X_train, y_train = train_df['text_clean'], train_df['HS']
X_dev, y_dev = dev_df['text_clean'], dev_df['HS']
X_test, y_test = test_df['text_clean'], test_df['HS']

print(f"Train: {len(X_train)} tweets | Dev: {len(X_dev)} tweets | Test: {len(X_test)} tweets")

## Búsqueda de hiperparámetros usando el split de validación

Armamos un pipeline con:

1. **Vectorización TF**: `CountVectorizer` con ngramas de 1 y 2 palabras
2. **Regresión logística LASSO** (penalización L1, solver `liblinear`)

Para elegir el mejor valor de `C` (inverso de la fuerza de regularización) probamos 30 valores en escala logarítmica entre $10^{-10}$ y $10^{1}$: para cada uno, entrenamos el pipeline sobre `train` y calculamos el ROC AUC sobre `dev`. Nos quedamos con el `C` que da mejor ROC AUC en `dev`.

In [ ]:
# Creamos una grilla de valores para el parámetro C (inverso de la penalización)
C_values = np.logspace(-10, 1, 30)

# Pipeline con TF y LASSO
pipeline_tf = Pipeline([
    ('vectorizer', CountVectorizer(
        ngram_range=(1, 2),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

# Almacenaremos los resultados de la búsqueda aquí
val_results = []

In [ ]:
%%time
# Para cada valor de C
for C in C_values:
    pipeline_tf.set_params(classifier__C=C)

    # Entrenamos sobre train y evaluamos sobre dev
    pipeline_tf.fit(X_train, y_train)
    dev_proba = pipeline_tf.predict_proba(X_dev)[:, 1]

    val_results.append({
        'C': C,
        'roc_auc_dev': roc_auc_score(y_dev, dev_proba)
    })

# Convertimos resultados a DataFrame
val_results_df = pd.DataFrame(val_results)

# Encontramos el mejor C
best_idx = val_results_df['roc_auc_dev'].idxmax()
best_C_tf = val_results_df.loc[best_idx, 'C']

print(f"Mejor valor de C: {best_C_tf:.6f}")
print(f"Mejor ROC AUC (dev): {val_results_df.loc[best_idx, 'roc_auc_dev']:.3f}")

## Modelo final

Entrenamos el pipeline final sobre `train` con el mejor valor de `C` encontrado y evaluamos sobre `test` (que no participó ni del entrenamiento ni de la elección de hiperparámetros) con ROC AUC, accuracy, precision, recall y F1.

In [ ]:
# Ajustamos el modelo final con el mejor C
final_pipeline_tf = Pipeline([
    ('vectorizer', CountVectorizer(
        ngram_range=(1, 2),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        C=best_C_tf,
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

final_pipeline_tf.fit(X_train, y_train)

# Predicciones en el conjunto de test
y_pred = final_pipeline_tf.predict(X_test)
y_pred_proba = final_pipeline_tf.predict_proba(X_test)[:, 1]

# Métricas finales
results_tf = {
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

for metric, value in results_tf.items():
    print(f"{metric}: {value:.3f}")

# TF-IDF con LASSO

A diferencia de TF (que solo cuenta ocurrencias), **TF-IDF** pondera cada término según qué tan frecuente es dentro de un tweet (TF) pero penalizando los términos que aparecen en muchos tweets distintos (IDF). Esto suele darle más peso a palabras discriminativas y menos a palabras muy comunes.

Reutilizamos los mismos `X_train`/`y_train`, `X_dev`/`y_dev` y `X_test`/`y_test` de la sección anterior, y repetimos el mismo procedimiento (elegir `C` con el split de validación y evaluar sobre test) pero con `TfidfVectorizer` en lugar de `CountVectorizer`.

In [ ]:
# Pipeline con TF-IDF y LASSO
pipeline_tfidf = Pipeline([
    ('vectorizer', TfidfVectorizer(
        ngram_range=(1, 2),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

val_results = []

In [ ]:
%%time
for C in C_values:
    pipeline_tfidf.set_params(classifier__C=C)

    pipeline_tfidf.fit(X_train, y_train)
    dev_proba = pipeline_tfidf.predict_proba(X_dev)[:, 1]

    val_results.append({
        'C': C,
        'roc_auc_dev': roc_auc_score(y_dev, dev_proba)
    })

val_results_df = pd.DataFrame(val_results)

best_idx = val_results_df['roc_auc_dev'].idxmax()
best_C_tfidf = val_results_df.loc[best_idx, 'C']

print(f"Mejor valor de C: {best_C_tfidf:.6f}")
print(f"Mejor ROC AUC (dev): {val_results_df.loc[best_idx, 'roc_auc_dev']:.3f}")

In [ ]:
# Ajustamos el modelo final con el mejor C
final_pipeline_tfidf = Pipeline([
    ('vectorizer', TfidfVectorizer(
        ngram_range=(1, 2),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        C=best_C_tfidf,
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

final_pipeline_tfidf.fit(X_train, y_train)

y_pred = final_pipeline_tfidf.predict(X_test)
y_pred_proba = final_pipeline_tfidf.predict_proba(X_test)[:, 1]

results_tfidf = {
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

for metric, value in results_tfidf.items():
    print(f"{metric}: {value:.3f}")

# Word embeddings como features

## Idea general

En lugar de representar cada tweet como un vector disperso de conteos (TF) o pesos (TF-IDF) sobre el vocabulario, ahora vamos a representarlo como el **promedio de los vectores de embedding preentrenados** de sus palabras. Usamos los embeddings estáticos en español **SBWCE** (`SBW-vectors-300-min5`, ya descargados al inicio del notebook), donde cada palabra se representa con un vector denso de 300 dimensiones.

El preprocesamiento acá es más simple: solo eliminamos URLs y menciones, y reemplazamos números por `DIGITO`. No hace falta sacar puntuación ni acentos porque las palabras que no estén en el vocabulario de los embeddings simplemente se ignoran (*out-of-vocabulary*).

Como en las secciones anteriores, vectorizamos por separado los tweets en español de `train`, `dev` y `test`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import re
from gensim.models import KeyedVectors
import nltk
from nltk.tokenize import word_tokenize
import warnings
warnings.filterwarnings('ignore')

# Descargas necesarias para tokenización
nltk.download('punkt_tab')
nltk.download('punkt')

# Preprocesamiento simple: solo removemos URLs, menciones y reemplazamos dígitos
def preprocess_text_embed(text):
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'\d+', 'DIGITO', text)
    return text

# Cargamos un split y nos quedamos con los tweets en español
def load_hateval_split_embed(path):
    df = pd.read_csv(path)
    df = df[df['language'] == 'es'].reset_index(drop=True)
    df['text_clean'] = df['text'].apply(preprocess_text_embed)
    return df

train_df = load_hateval_split_embed(train_path)
dev_df = load_hateval_split_embed(dev_path)
test_df = load_hateval_split_embed(test_path)

# Cargamos el modelo de word embeddings
def load_embeddings(path):
    print("Cargando embeddings...")
    return KeyedVectors.load_word2vec_format(path, binary=True)

word_vectors = load_embeddings("./models/SBW-vectors-300-min5.bin")

# Función para obtener el vector promedio de un tweet
def get_mean_vector(text, word_vectors, vector_size=300):
    words = word_tokenize(text.lower())  # Tokenizamos y convertimos a minúsculas
    word_vectors_list = []

    for word in words:
        try:
            vector = word_vectors[word]
            word_vectors_list.append(vector)
        except KeyError:
            continue  # Ignoramos palabras que no están en el embedding

    if word_vectors_list:
        return np.mean(word_vectors_list, axis=0)
    else:
        return np.zeros(vector_size)  # Vector de ceros si no hay palabras válidas

# Convertimos los tweets de un split a vectores
def vectorize_split(df, word_vectors):
    print("Vectorizando tweets...")
    vectors = [get_mean_vector(text, word_vectors) for text in df['text_clean']]
    X = pd.DataFrame(vectors, columns=[f'V{i+1}' for i in range(300)])
    X['id'] = df['id'].values
    y = df['HS'].values
    return X, y

X_train_embed_full, y_train = vectorize_split(train_df, word_vectors)
X_dev_embed_full, y_dev = vectorize_split(dev_df, word_vectors)
X_test_embed_full, y_test = vectorize_split(test_df, word_vectors)

# Guardamos los ids por separado y nos quedamos solo con las columnas de vectores
train_ids = X_train_embed_full['id']
dev_ids = X_dev_embed_full['id']
test_ids = X_test_embed_full['id']
X_train_embed = X_train_embed_full.drop('id', axis=1)
X_dev_embed = X_dev_embed_full.drop('id', axis=1)
X_test_embed = X_test_embed_full.drop('id', axis=1)

In [ ]:
X_train_embed.head()

## Búsqueda de hiperparámetros usando el split de validación

Repetimos el mismo esquema que en las secciones anteriores: para cada valor de `C` entrenamos sobre `X_train_embed`/`y_train` y evaluamos el ROC AUC sobre `X_dev_embed`/`y_dev`, y nos quedamos con el que mejor performa en `dev`.

In [ ]:
# Grilla de valores para C
C_values = np.logspace(-10, 1, 30)

val_results = []

In [ ]:
%%time
print("Buscando el mejor valor de C...")
for C in C_values:
    model = LogisticRegression(
        C=C,
        penalty='l1',
        solver='liblinear',
        random_state=234
    )

    model.fit(X_train_embed, y_train)
    dev_proba = model.predict_proba(X_dev_embed)[:, 1]

    val_results.append({
        'C': C,
        'roc_auc_dev': roc_auc_score(y_dev, dev_proba)
    })

val_results_df = pd.DataFrame(val_results)

best_idx = val_results_df['roc_auc_dev'].idxmax()
best_C_embed = val_results_df.loc[best_idx, 'C']

print(f"\nMejor valor de C: {best_C_embed:.6f}")
print(f"Mejor ROC AUC (dev): {val_results_df.loc[best_idx, 'roc_auc_dev']:.3f}")

In [ ]:
# Ajustamos el modelo final con el mejor C
final_model_embed = LogisticRegression(
    C=best_C_embed,
    penalty='l1',
    solver='liblinear',
    random_state=234
)

final_model_embed.fit(X_train_embed, y_train)

y_pred = final_model_embed.predict(X_test_embed)
y_pred_proba = final_model_embed.predict_proba(X_test_embed)[:, 1]

results_embed = {
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

print("\nResultados finales:")
for metric, value in results_embed.items():
    print(f"{metric}: {value:.3f}")

# Comparación de resultados

In [ ]:
results_tf_df = pd.DataFrame([results_tf], index=['TF'])
results_tfidf_df = pd.DataFrame([results_tfidf], index=['TF-IDF'])
results_embed_df = pd.DataFrame([results_embed], index=['Word Embeddings'])

comparison_df = pd.concat([results_tf_df, results_tfidf_df, results_embed_df])

print("Comparación de resultados de los modelos (evaluados sobre test):")
display(comparison_df)

# Ejercicio

Entrenar y tunear un Random Forest usando features construidas con TF-IDF y otro con embeddings, para clasificar los tweets en español del dataset HatEval. Usá el mismo esquema que en este notebook: elegí los hiperparámetros con el split de `dev` y reportá el desempeño final sobre `test`.

¿Cuál resulta más eficiente? ¿Por qué?

Como desafío adicional: ¿qué esperarías que pase si usás estos mismos embeddings en español (SBWCE) para vectorizar los tweets en **inglés** del dataset (`language == 'en'`)?

In [ ]:
###